# Meal Planning Agent — Prompt Experiments

Scratchpad for iterating on prompts before building the app.

## Initialization

In [69]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings, OpenAIChatCompletionsModel
from agents.extensions.visualization import draw_graph
from openai import AsyncOpenAI
from pydantic import BaseModel, Field
import os
import asyncio

load_dotenv(override=True)

def assertKeyExists(key :str) -> str:
    value = os.getenv(key)
    if value:
        return value
    raise ValueError(f"{key} not found")

openai_api_key = assertKeyExists("OPENAI_API_KEY")
google_api_key = assertKeyExists("GOOGLE_API_KEY")
grok_api_key = assertKeyExists("GROK_API_KEY")

print("API keys loaded ✔")

high_effort_model = "gpt-5.6-sol"
default_model = "gpt-5.6-luna"
low_effort_model = "gpt-5.6-terra"

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.6-flash", openai_client=gemini_client)

GROK_BASE_URL = "https://api.x.ai/v1"
grok_client = AsyncOpenAI(base_url=GROK_BASE_URL, api_key=grok_api_key)
grok_model = OpenAIChatCompletionsModel(model="grok-4.5", openai_client=grok_client)

print("Models loaded ✔")

base_system_instructions = '''
You are a meal planning assistant who helps people plan their meals for the week.
You are an expert on simple meals that require minimal amounts of preparation, reheat well, and are delicious.
'''

def to_markdown_list(data: list[any], bullet: str ="-"):
    """
    Converts a list into a markdown bulleted list string.
    """
    return "\n".join(f"{bullet} {str(item)}" for item in data)

def filter_out_invalid_strings(input_list: list[str], invalid_strings: set[str]) -> list[str]:
    [item for item in input_list if item not in invalid_strings]

print("Utils loaded ✔")

API keys loaded ✔
Models loaded ✔
Utils loaded ✔


In [ ]:
pushover_user = assertKeyExists("PUSHOVER_USER")
pushover_token = assertKeyExists("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


def send_push_notification(message: str):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

## User Preferences 
Collects dietary restrictions, likes/dislikes, meals that they're tired of, and what kind of cooking equipment they have. 

In [70]:
class UserPreferences(BaseModel):
    number_of_meals: int = Field(default = 2, description="The number of meals that you should plan.")
    dietary_restrictions: list[str] = Field(default_factory=list, description="A list of any dietary restrictions the user has that need to be considered for the meal plan.", examples=["Gluten-free", "vegetarian", "dairy-free"])
    likes: list[str] = Field(default_factory=list, description="A list of foods user's favorite foods.")
    dislikes: list[str] = Field(default_factory=list, description="A list of foods that the user does not like.")
    nutritional_goals: str = Field(default="", description="A description of the nutritional goals that the user aims to achieve with this meal plan.", examples=["Increase protein intake, lose weight, lower cholesterol"])
    meals_to_avoid_this_time: list[str] = Field(default_factory=list, description="Specific foods that the user would prefer to avoid this plan.")
    notes: str = Field(default="", description="A paragraph of notes about any preferences that don't apply to one of the other fields.", examples=["Half of my meals should be meatless."])

class UserPreferencesReview(BaseModel):
    preferences: UserPreferences = Field(description="The user's updated preferences.")
    user_has_confirmed_that_preferences_are_correct: bool = Field(description="True only when the user has reviewed their preferences, confirmed that they are correct, and that they don't want to modify it any further.")
    follow_up_response: str = Field(description="Your polite response to the user's message confirming that you understand their request. If the user has not yet confirmed that the preferences are complete, ask them if there is anything else.")

saved_user_preferences = UserPreferences()

def set_user_preferences(update: UserPreferences):
    global saved_user_preferences
    saved_user_preferences = update

def get_user_preferences() -> UserPreferences:
    return saved_user_preferences

user_preferences_system_instructions = f'''
{base_system_instructions}
'''

user_preferences_prompter = Agent(
    name="User Preferences",
    instructions=user_preferences_system_instructions,
    model=default_model,
)

user_preferences_reviewer = Agent(
    name="User Preferences",
    instructions=user_preferences_system_instructions,
    model=default_model,
    output_type=UserPreferencesReview,
)

async def present_known_user_preferences() -> str:
    prompt =f'''
    Here is all of the information we have about the user's meal plan preferences:
    {get_user_preferences()}

    Present it to the user in an easily reviewable format.

    You need the correct preferences to make sure that the meal plan works for the user. It's essential for you to do your job correctly.
    Explain the importance to the user.
    Then ask the user to either confirm that everything looks correct or make changes.
    '''

    return (await Runner.run(user_preferences_prompter, prompt)).final_output

async def update_user_preferences(user_update: str) -> UserPreferencesReview:
    prompt =f'''
    Here is all of the information we have about the user's meal plan preferences:
    {get_user_preferences()}

    You presented this information to the user.

    The user sent this message:
    {user_update}

    If the user has not yet confirmed that the preferences are correct and complete, prompt them to finish it in the follow_up_response field.
    '''

    return (await Runner.run(user_preferences_reviewer, prompt)).final_output


In [ ]:
with trace("Empty User Preferences Demo"):
    set_user_preferences(UserPreferences())
    print(await present_known_user_preferences())

In [ ]:
with trace("Happy Path User Preferences Demo"):
    set_user_preferences(UserPreferences(dietary_restrictions=["egg allergy"],nutritional_goals="Lose weight, High protein",notes="I want half my meals to be meatless"))
    print(await present_known_user_preferences())
    print("\n\n")

    first_user_response = "I want half of my meals to be meatless. I want my food to be relatively healthy too so I can try to lose wait as well."
    print(f"{first_user_response}\n\n")
    first_update = await update_user_preferences(first_user_response)
    print(f"{first_update}\n\n")
    

    second_user_response = "That looks correct, thank you."
    print(f"{second_user_response}\n\n")
    second_update = await update_user_preferences(second_user_response)
    print(f"{second_update}\n\n")


## Meal Generation

### Meal Brainstorming Agent

An agent that generates a bunch of meal ideas.

In [ ]:
from datetime import datetime

# Used to make the meals weather-appropriate and add a little bit of differentiation to the prompt week-to-week.
def get_seasonal_report():
    now = datetime.now()
    month_name = now.strftime("%B")
    month_num = now.month
    
    # Season list (indexed 0-3)
    seasons = ["Winter", "Spring", "Summer", "Autumn"]
    
    # Weather descriptions for each season
    weather_data = {
        "Winter": "Expect cold temperatures, frosty mornings, and the occasional flurry of snow.",
        "Spring": "The days are getting longer and you'll see flowers beginning to bloom.",
        "Summer": "It's time for sunshine, warm breeze, and plenty of outdoor activities.",
        "Autumn": "The air is turning crisp and the leaves are putting on a colorful show."
    }
    
    # The math trick: (month % 12 // 3)
    # Dec(12), Jan(1), Feb(2) map to 0 (Winter)
    # Mar(3), Apr(4), May(5) map to 1 (Spring)
    # Jun(6), Jul(7), Aug(8) map to 2 (Summer)
    # Sep(9), Oct(10), Nov(11) map to 3 (Autumn)
    season_idx = (month_num % 12 // 3)
    season = seasons[season_idx]
    description = weather_data[season]
    
    return f"It's {month_name} and {season} is here! {description}"

# Output the result
print(get_seasonal_report())

In [72]:
import random

# Randomly returns a model so that the behavior is more variable
def get_random_model():
    models = [default_model, gemini_model, grok_model]
    return random.choice(models)

In [101]:
class PreparedDish(BaseModel):
    name: str = Field(description="The short name of a dish.")
    description: str = Field(description="A 1-2 sentence description of the dish.")
    special_diet_labels: list[str] = Field(description="A list of any dietary restrictions that this meal satisfies", examples=["vegetarian", "gluten-free"])
    category: list[str] = Field(description="The category of food.", examples = ["Italian", "Chinese"])


class MealPlanIdeas:
    entree_ideas: list[PreparedDish]
    side_ideas: list[PreparedDish] 

    def __init__(self, entree_ideas: list[PreparedDish], side_ideas: list[PreparedDish]) -> None:
        self.entree_ideas = entree_ideas
        self.side_ideas = side_ideas

    def __str__(self):
        return f"entrees: {self.entree_ideas}, sides: {self.side_ideas}"


brainstorm_instructions = f'''
{base_system_instructions}

{get_seasonal_report()}
Try to pick meals that are popular for this time of year.
'''

meal_brainstorming_agent = Agent(
    name="Meal Brainstormer",
    instructions=brainstorm_instructions,
    model=get_random_model(),
    output_type=list[PreparedDish],
)

async def generate_meal_ideas(prompt: str) -> list[PreparedDish]:
    return (await Runner.run(meal_brainstorming_agent, prompt)).final_output


async def create_meal_plan_brainstorm() -> MealPlanIdeas:
    preferences = get_user_preferences()
    number_of_meals = preferences.number_of_meals
    user_preferences_prompt = f'''
    Here is the user's preferences:
    {preferences}
    '''
    
    meal_idea_multiple = 5 # generate extra ideas to allow for more randomness and also in case we need to drop some of them during validation
    entrees, sides = await asyncio.gather(
        generate_meal_ideas(f"Suggest {number_of_meals * meal_idea_multiple} different entrees. {user_preferences_prompt}"),
        generate_meal_ideas(f"Suggest {number_of_meals * meal_idea_multiple} different sides. {user_preferences_prompt}")
    )
    return MealPlanIdeas(
        entree_ideas = entrees, 
        side_ideas = sides,
    )

In [ ]:
with trace("Meal Brainstorming Test"):
    set_user_preferences(UserPreferences(nutritional_goals="Lose weight, High protein",notes="I want half my meals to be meatless"))
    unfiltered_meal_plan = await create_meal_plan_brainstorm()
    print(unfiltered_meal_plan)

### Validation
Filters brainstorm ideas that don't conform to likes, dislikes, and user preferences.
Verifies that it's not too repetitive with the meals from last week.

In [102]:
meal_validation_instructions = f'''
{base_system_instructions}

Part of your job is inspecting menus for clients and flagging any foods that they would dislike.
Identifying violations of your client's food allergen or dietary restriction rules is your highest priority.
'''

meal_filterer = Agent[any](
    name = "Meal Idea Filterer",
    instructions=meal_validation_instructions,
    model = default_model,
    output_type = set[str],
)

async def filter_meal_ideas(meal_ideas: MealPlanIdeas) -> MealPlanIdeas:
    prompt = f'''
    You have a list of foods that have been generated as candidates for the user's meal plan:
    {to_markdown_list(meal_ideas.entree_ideas + meal_ideas.side_ideas)}

    Here is the user's meal plan preferences:
    {get_user_preferences()}

    Your job is to identify and return any of those foods that are poor candidates based on the user's preferences.
    '''
    flagged_foods = (await Runner.run(meal_filterer, prompt)).final_output

    # Filter out flagged foods and shuffle them to make the selection more random.
    return MealPlanIdeas(
        entree_ideas=filter_out_invalid_strings(meal_ideas.entree_ideas, flagged_foods),
        side_ideas=filter_out_invalid_strings(meal_ideas.side_ideas, flagged_foods),
    )

In [ ]:
with trace("Meal Validation Test"):
    set_user_preferences(UserPreferences(dietary_restrictions=["gluten-free"]))
    result = await filter_meal_ideas(
        MealPlanIdeas(
            entree_ideas=[PreparedDish("Spaghetti and Meatballs", "", "", [])],
            side_ideas=[PreparedDish("Macaroni and Cheese", "", "", [])],
        )
    )
    print(result)

### Pairing
Picks meals and attempts to pair it with similar side based on category.

In [114]:
import random

class MealPairing(BaseModel):
    entree: PreparedDish = Field(description="main entree")
    side: PreparedDish = Field(description="side")

pairing_system_instructions=f'''
{base_system_instructions}
'''
entree_picking_agent=Agent[any](
    name="Entree Picking Agent",
    instructions=pairing_system_instructions,
    model=default_model,
    output_type=list[PreparedDish]
)
pairing_agent=Agent[any](
    name="Meal Pairing Agent",
    instructions=pairing_system_instructions,
    model=default_model,
    output_type=list[MealPairing]
)

meal_choice_validation_agent=Agent[any](
    name="Meal Choice Validation Agent",
    instructions=pairing_system_instructions,
    model=gemini_model,
    output_type=bool
)

async def generate_meals(brainstorm_results: MealPlanIdeas) -> list[MealPairing]:
    num_meals = get_user_preferences().number_of_meals
    attempts = 0
    while(True):
        attempts += 1
        entree_choices = await pick_entrees(num_meals = num_meals, entrees = brainstorm_results.entree_ideas)
        print(f"\nnumber of entrees: {len(entree_choices)}\n")
        meal_choices = await pair_with_sides(entrees = entree_choices, sides = brainstorm_results.side_ideas)
        print(f"\nnumber of meals: {len(meal_choices)}\n")
        if (await validate_meal_choices(meal_choices) or attempts > 3):
            return meal_choices

async def pick_entrees(num_meals: int, entrees: list[PreparedDish]) -> list[PreparedDish]:
    # Shuffle to make meal selection more unpredictable
    shuffled_entrees = random.sample(entrees, len(entrees))
    prompt=f'''
    Your job is to pick {num_meals} entree(s) for the user's meal plan.

    You can choose from the following list:
    {to_markdown_list(shuffled_entrees)}

    Make sure that your final selection conforms to the user's preferences:
    {get_user_preferences()}
    '''
    return (await Runner.run(entree_picking_agent, prompt)).final_output


async def pair_with_sides(entrees: list[PreparedDish], sides: list[PreparedDish]) -> list[MealPairings]:
    # Shuffle to make meal selection more unpredictable
    shuffled_sides = random.sample(sides, len(sides))
    prompt = f'''
    You're writing a meal plan for the user.

    You already have the entrees picked out:
    {to_markdown_list(entrees)}

    Now you need to pair those entrees with sides so that you have a complete meal.
    You can choose from the following list of sides:
    {to_markdown_list(shuffled_sides)}

    For each entree, try to pick a side that pairs well with it.
    Ideally they should be the same or similar categories.
    '''
    return (await Runner.run(pairing_agent, prompt)).final_output

async def validate_meal_choices(meals: list[MealPairing]) -> bool:
    prompt = f'''
    You're writing a meal plan for the user.

    You've picked meal(s) for the meal plan':
    {to_markdown_list(meals)}

    Determine if that list conforms to the user's preferences:
    {get_user_preferences()}
    '''
    return (await Runner.run(meal_choice_validation_agent, prompt)).final_output


### User Feedback
Asks user to approve the selected meals. Can make changes to a meal or select a different one from the brainstorming list. If all else fails, can escape back and restart with a different model.

In [115]:
meal_plan_feedback_system_prompt =f'''
{base_system_instructions}
'''
meal_plan_feedback_agent = Agent[any](
    name = "Meal Plan Feedback Agent",
    instructions = meal_plan_feedback_system_prompt,
    model = default_model,
)

async def present_meals_for_feedback(meal_pairings: list[MealPairing]) -> str:
    prompt=f'''
    You have successfully generated meals for the user's meal plan!
    Here they are:
    {to_markdown_list(meal_pairings)}

    Present all of them to the user in markdown for their review.

    For your tone, be polite and don't be afraid to embellish how tasty these meals are going to be.
    Ask them to approve the meals or request changes before you begin generating recipes.
    '''

    return (await Runner.run(meal_plan_feedback_agent, prompt)).final_output

### Demo
Ties the meal generation flow altogether

In [ ]:
with trace("Meal Plan Generation Demo"):
    set_user_preferences(UserPreferences(nutritional_goals="Lose weight, Increase protein intake",notes="I want half my meals to be meatless"))
    brainstorm_results = await create_meal_plan_brainstorm()
    print("Brainstorm Results:\n")
    print(brainstorm_results)
    meal_pairings = await generate_meals(brainstorm_results)
    print("\n\nMeal Pairings:\n")
    print(meal_pairings)
    print("\n\nMeal Plan:\n")
    print(await present_meals_for_feedback(meal_pairings))

## Recipes

### Generation
Writes recipes for the selected meals

### Validation
Makes sure that the recipes conform to the user's dietary restrictions and can be made with the user's equipment.

## Shopping List

### Aggregation
Collects all of the ingredients from the recipes above

### Formatting
Combines identical ingredients and groups them by section of the super market

## Meal Plan Presentation Agent
Turns everything into pretty markdown:
* High level plan
* Shopping List
* Recipes

## Orchestration
Ties everything together and runs it in a UI.

### Basic Sequential Iteration
Goes through steps one by one.

In [ ]:
with trace("Meal Planning"):
    unfiltered_meal_plan = create_meal_plan_brainstorm(2)
    print(unfiltered_meal_plan)

### LLM Orchestration
Use tools and subagents to allow better flexibility like moving backwards to change preferences or meal choices